In [2]:
import pandas as pd
from selenium.webdriver.common.by import By
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from utils.constants import CHROMEDRIVER_PATH, CHROMEDRIVER_DESKTOP_PATH, UU_REGULATIONS_CSV, UU_REGULATION_DETAILS_CSV, UU_DATA_DIR, UU_METADATA
from utils.scraping import download_file, generate_id, create_metadata

In [2]:
service = Service(executable_path=CHROMEDRIVER_DESKTOP_PATH)
driver = webdriver.Chrome(service=service)

# Scrape List of Regulations

In [13]:
regulations_xpath = '//div[@class="col-md-12"]'
# title_xpath = './/p[@style="padding-top: -2;"]'
title_xpath = './/p[@style="font-size: 14px;"]'
link_xpath = './/a[@title="lihat detail"]'

In [14]:
MAX_PAGES = 91
WEBSITE = "https://peraturan.go.id/uu?page="

all_regulations = []

for page in range(1, MAX_PAGES+1):
  web_page = WEBSITE + str(page)
  driver.get(web_page)
  print("web_page", web_page)

  WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, regulations_xpath)))
  
  regulations = driver.find_elements(By.XPATH, regulations_xpath)

  for regulation in regulations:
    title = regulation.find_element(By.XPATH, title_xpath)
    link = regulation.find_element(By.XPATH, link_xpath)

    all_regulations.append({"title": title.text, "link": link.get_attribute('href')})

web_page https://peraturan.go.id/uu?page=1
web_page https://peraturan.go.id/uu?page=2
web_page https://peraturan.go.id/uu?page=3
web_page https://peraturan.go.id/uu?page=4
web_page https://peraturan.go.id/uu?page=5
web_page https://peraturan.go.id/uu?page=6
web_page https://peraturan.go.id/uu?page=7
web_page https://peraturan.go.id/uu?page=8
web_page https://peraturan.go.id/uu?page=9
web_page https://peraturan.go.id/uu?page=10
web_page https://peraturan.go.id/uu?page=11
web_page https://peraturan.go.id/uu?page=12
web_page https://peraturan.go.id/uu?page=13
web_page https://peraturan.go.id/uu?page=14
web_page https://peraturan.go.id/uu?page=15
web_page https://peraturan.go.id/uu?page=16
web_page https://peraturan.go.id/uu?page=17
web_page https://peraturan.go.id/uu?page=18
web_page https://peraturan.go.id/uu?page=19
web_page https://peraturan.go.id/uu?page=20
web_page https://peraturan.go.id/uu?page=21
web_page https://peraturan.go.id/uu?page=22
web_page https://peraturan.go.id/uu?page=

In [15]:
df = pd.DataFrame(all_regulations)
df.to_csv(UU_REGULATIONS_CSV, index=False)

# Scrape Regulation Detail

In [19]:
title_div_xpath = '//div[@class="detail_title_1"]'
table_xpath = '//table[@class="table caption-top"]'
row_xpath = './/tr'
row_q_xpath = './/th[@style="width: 30%; font-size: 14px;"]'
row_a_xpath = './/td'
docs_xpath = './/a'

SKIP_COLS = ["pejabat_yang_menetapkan", "tahun_pengundangan", "nomor_pengundangan",
              "nomor_tambahan", "tanggal_pengundangan", "pejabat_pengundangan"]

In [20]:
df_all_reg = pd.read_csv(UU_REGULATIONS_CSV)

In [21]:
from utils.scraping import slugify

all_regulation_details = []

for index, row in df_all_reg.iterrows():
  print(index, row['link'])
  reg_detail = {}
  driver.get(row['link'])
  WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.XPATH, table_xpath)))

  title = driver.find_element(By.XPATH, title_div_xpath)
  reg_detail["title"] = title.text

  tables = driver.find_elements(By.XPATH, table_xpath)

  for table in tables:
    rows = table.find_elements(By.XPATH, row_xpath)
    for row in rows:
      row_q = row.find_element(By.XPATH, row_q_xpath)
      row_q_text = slugify(row_q.text)

      if row_q_text == "dokumen_peraturan":
        row_as = row.find_elements(By.XPATH, docs_xpath)
        row_a_texts = [row_a.get_attribute('href') for row_a in row_as]
      elif row_q_text in SKIP_COLS:
        continue
      else:
        row_a = row.find_element(By.XPATH, row_a_xpath)
        row_a_texts = row_a.text
        
      reg_detail[row_q_text] = row_a_texts
  
  all_regulation_details.append(reg_detail)

0 https://peraturan.go.id/id/uu-no-53-tahun-2024
1 https://peraturan.go.id/id/uu-no-36-tahun-2024
2 https://peraturan.go.id/id/uu-no-31-tahun-2024
3 https://peraturan.go.id/id/uu-no-25-tahun-2024
4 https://peraturan.go.id/id/uu-no-57-tahun-2024
5 https://peraturan.go.id/id/uu-no-49-tahun-2024
6 https://peraturan.go.id/id/uu-no-20-tahun-2024
7 https://peraturan.go.id/id/uu-no-18-tahun-2024
8 https://peraturan.go.id/id/uu-no-39-tahun-2024
9 https://peraturan.go.id/id/uu-no-19-tahun-2024
10 https://peraturan.go.id/id/uu-no-17-tahun-2024
11 https://peraturan.go.id/id/uu-no-51-tahun-2024
12 https://peraturan.go.id/id/uu-no-40-tahun-2024
13 https://peraturan.go.id/id/uu-no-22-tahun-2024
14 https://peraturan.go.id/id/uu-no-55-tahun-2024
15 https://peraturan.go.id/id/uu-no-32-tahun-2024
16 https://peraturan.go.id/id/uu-no-9-tahun-2024
17 https://peraturan.go.id/id/uu-no-38-tahun-2024
18 https://peraturan.go.id/id/uu-no-14-tahun-2024
19 https://peraturan.go.id/id/uu-no-4-tahun-2024
20 https://p

In [22]:
df = pd.DataFrame(all_regulation_details)
df.to_csv(UU_REGULATION_DETAILS_CSV, index=False)

In [ ]:
## Note:
# Hubungan Antar Peraturan & Dasar Hukum not scraped yet

# Download Files

In [3]:
import ast

df = pd.read_csv(UU_REGULATION_DETAILS_CSV)
df['dokumen_peraturan'] = df['dokumen_peraturan'].apply(ast.literal_eval)

In [4]:
df.head(1)

,title,jenis_bentuk_peraturan,pemrakarsa,nomor,tahun,tentang,tempat_penetapan,ditetapkan_tanggal,status,dokumen_peraturan,jumlah_dilihat,jumlah_didownload
0,Undang-undang Nomor 53 Tahun 2024 Tentang Kota...,UNDANG-UNDANG,PEMERINTAH PUSAT,53,2024,KOTA BUKITTINGGI DI PROVINSI SUMATERA BARAT,Jakarta,01 Januari 1970,Berlaku,[https://peraturan.go.id/files/uu-no-53-tahun-...,756,606.0


In [5]:
for row in df['dokumen_peraturan']:
  print(row)

['https://peraturan.go.id/files/uu-no-53-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-36-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-31-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-25-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-57-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-49-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-20-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-18-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-39-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-19-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-17-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-51-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-40-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-22-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-55-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-32-tahun-2024.pdf']
['https://peraturan.go.id/files/uu-no-9-tahun-2024.pdf']
['https://perat

In [7]:
import os
from requests.exceptions import RequestException
from utils.scraping import slugify

download_dir = UU_DATA_DIR
metadata_list = []

for index, row in df.iterrows():
  if not row['dokumen_peraturan']:
    print(f"No files to download for row {index}")
    continue
    
  slugified_date = slugify(row['ditetapkan_tanggal'])
    
  for doc_url in row['dokumen_peraturan']:
      original_filename = os.path.basename(doc_url)
      filename, extension = os.path.splitext(original_filename)

      unique_id = generate_id()
      slugified_filename = slugify(filename)
      new_filename = f"uu_{slugified_date}_{slugified_filename}_{unique_id}{extension}"
      new_filepath = os.path.join(download_dir, new_filename)

      try:
          download_file(doc_url, new_filepath)
          print(f"Downloaded {doc_url} to {new_filepath}")
      except RequestException as e:
          print(f"Failed to download {doc_url}: {e}")
          continue
      
      metadata = create_metadata(row, doc_url, unique_id, new_filename)
      metadata_list.append(metadata)

      # if only downloading one file per row
      # break

Downloaded https://peraturan.go.id/files/uu-no-53-tahun-2024.pdf to data_pool/uu\uu_01_januari_1970_uu_no_53_tahun_2024_omizfp2f.pdf
Downloaded https://peraturan.go.id/files/uu-no-36-tahun-2024.pdf to data_pool/uu\uu_01_januari_1970_uu_no_36_tahun_2024_wsiajowk.pdf
Downloaded https://peraturan.go.id/files/uu-no-31-tahun-2024.pdf to data_pool/uu\uu_02_juli_2024_uu_no_31_tahun_2024_ewlyfrqj.pdf
Downloaded https://peraturan.go.id/files/uu-no-25-tahun-2024.pdf to data_pool/uu\uu_02_juli_2024_uu_no_25_tahun_2024_cvxuwrh4.pdf
Downloaded https://peraturan.go.id/files/uu-no-57-tahun-2024.pdf to data_pool/uu\uu_01_januari_1970_uu_no_57_tahun_2024_quxwauvw.pdf
Downloaded https://peraturan.go.id/files/uu-no-49-tahun-2024.pdf to data_pool/uu\uu_01_januari_1970_uu_no_49_tahun_2024_cqbi8x5v.pdf
Downloaded https://peraturan.go.id/files/uu-no-20-tahun-2024.pdf to data_pool/uu\uu_02_juli_2024_uu_no_20_tahun_2024_pqobbbz2.pdf
Downloaded https://peraturan.go.id/files/uu-no-18-tahun-2024.pdf to data_pool/

In [8]:
import json

with open(UU_METADATA, 'w') as json_file:
    json.dump(metadata_list, json_file, indent=4)

print("All files downloaded and metadata saved to JSON.")

All files downloaded and metadata saved to JSON.
